# ARC2 Speed Test: Qwen3.8-27B serving throughput on 4x L4

Decision gate for the next ARC-AGI-2 plan: can any serving setup reach ~500 generated tokens/s in aggregate with real
ARC prompts and long thinking? Setups (priority order, skipped from the end if the ~85 min budget runs out):
vLLM 0.19 FP8 + MTP-1 (baseline) | vLLM 0.19 GPTQ W4A16 + MTP-1 | vLLM 0.30 FP8 + MTP-3 + FP8 KV |
vLLM 0.30 W4A16 + MTP-3 + FP8 KV | SGLang 0.5.20 FP8 + EAGLE-3 + FP8 KV.
Waves: 32 streams x 1024 tokens, 48 x 768, and 16 x 512 at ~16k-token contexts (thinking on, xhigh, ignore_eos).
Throughput is read from the servers' own logs (steady state) and also computed client-side (tokens / wall time).


In [ ]:
%%writefile bench.py
"""Serving speed test for Qwen3.8-27B on 4x L4 (ARC2 Speed Test notebook).

For each serving setup (backend x checkpoint x speculative decoding x KV dtype) it starts the server on all 4 GPUs,
checks it answers, then runs fixed-length generation waves with real ARC prompts and reads the server's own
throughput / acceptance / KV-usage log lines. One setup failing (install, start, OOM) only skips that setup.

    python bench.py <cfg.json>          -> /kaggle/working/bench_results.json + a printed table
"""
from __future__ import annotations

import glob
import json
import os
import re
import shutil
import signal
import statistics
import subprocess
import sys
import tarfile
import threading
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

T0 = time.time()
CFG: dict = {}
RES: dict = {"configs": [], "installs": {}, "events": []}


def log(*a):
    msg = " ".join(str(x) for x in a)
    print(f"[bench {time.time() - T0:7.1f}s] {msg}", flush=True)
    RES["events"].append(f"{time.time() - T0:.0f}s {msg}"[:400])


def save():
    path = os.path.join(CFG["work_dir"], "bench_results.json")
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(RES, f, indent=1)
    os.replace(tmp, path)


def find(pattern: str) -> str | None:
    hits = sorted(glob.glob(pattern, recursive=True))
    return hits[0] if hits else None


# ============================================================================ python 3.12 + one venv per stack
def setup_python() -> str:
    if CFG.get("mock"):
        return sys.executable
    py = "/tmp/py312/python/bin/python3.12"
    if not os.path.exists(py):
        tgz = find("/kaggle/input/**/cpython312.tgz.bin")
        os.makedirs("/tmp/py312", exist_ok=True)
        with tarfile.open(tgz, mode="r:gz") as tf:
            tf.extractall("/tmp/py312")
    return py


def install(py: str, stack: dict) -> dict:
    """Create /tmp/venv_<name> and install the stack's packages offline from the wheelhouse holding `wheel`."""
    name = stack["name"]
    st = {"ok": False, "t0": round(time.time() - T0)}
    t = time.time()
    try:
        if CFG.get("mock"):
            st.update(ok=True, python=sys.executable)
            return st
        free = shutil.disk_usage("/tmp").free / 2 ** 30
        st["free_gb_before"] = round(free, 1)
        if free < stack.get("need_gb", 14):
            raise RuntimeError(f"only {free:.0f} GB free in /tmp")
        whl = find(stack["wheel"])
        if not whl:
            raise RuntimeError(f"wheel not found: {stack['wheel']}")
        venv = f"/tmp/venv_{name}"
        shutil.rmtree(venv, ignore_errors=True)
        env = {"HOME": "/tmp", "TMPDIR": "/tmp", "PIP_NO_CACHE_DIR": "1"}
        rv = subprocess.run([py, "-m", "venv", venv], capture_output=True, text=True, timeout=600)
        if rv.returncode == 0:
            vpy = f"{venv}/bin/python"
            env["PATH"] = f"{venv}/bin:/usr/bin:/bin"
            target = []
        else:  # no venv support: install into a target dir and run the base interpreter with PYTHONPATH
            st["venv_error"] = (rv.stdout + rv.stderr)[-400:]
            vpy = py
            env["PATH"] = f"{os.path.dirname(py)}:/usr/bin:/bin"
            env["PYTHONPATH"] = venv
            target = ["--target", venv]
            st["pythonpath"] = venv
        pip = [vpy, "-m", "pip", "install", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
               "--find-links", os.path.dirname(whl)] + target
        r = subprocess.run(pip + stack["pkgs"], env=env, capture_output=True, text=True, timeout=3000)
        if r.returncode != 0:  # dependency names differ from what we guessed: install the whole wheelhouse
            st["pip_first_tail"] = (r.stdout + r.stderr)[-800:]
            wheels = sorted(glob.glob(os.path.join(os.path.dirname(whl), "*.whl")))
            r = subprocess.run(pip + wheels, env=env, capture_output=True, text=True, timeout=3000)
            st["installed_all_wheels"] = len(wheels)
        st["pip_rc"] = r.returncode
        st["pip_tail"] = (r.stdout + r.stderr)[-1500:]
        if r.returncode == 0:
            r2 = subprocess.run([vpy, "-c", stack.get("import_check", "import torch; print(torch.__version__)")], env=env,
                                capture_output=True, text=True, timeout=900)
            st["import"] = (r2.stdout + r2.stderr).strip()[-600:]
            st["ok"] = r2.returncode == 0
        st["python"] = vpy
    except Exception as e:
        st["error"] = repr(e)[:600]
    finally:
        st["s"] = round(time.time() - t)
    return st


def warm(paths: list):
    """Read model weights once so the first server start does not wait on /kaggle/input (~140 MB/s)."""
    for p in paths:
        for f in sorted(glob.glob(os.path.join(p, "*.safetensors"))):
            try:
                subprocess.run(["cat", f], stdout=subprocess.DEVNULL, timeout=1800)
            except Exception:
                pass


# ============================================================================ prompts (same format as qwen_stage)
def fmt(g) -> str:
    return f"{len(g)}x{len(g[0])}\n" + "\n".join(" ".join(str(int(v)) for v in r) for r in g)


SYSTEM = ("You are an expert at ARC-AGI puzzles and a precise Python programmer. Each puzzle has training examples "
          "(input grid -> output grid) that all follow ONE hidden rule. Find the rule and implement it as a Python "
          "function def transform(grid: list[list[int]]) -> list[list[int]]. Answer with one RULE: line, then one "
          "```python block.")


def task_prompt(t: dict) -> str:
    parts = [f"## Example {i}\nInput {fmt(p['input'])}\nOutput {fmt(p['output'])}" for i, p in enumerate(t["train"], 1)]
    parts += [f"## Test input {i}\n{fmt(x['input'])}" for i, x in enumerate(t["test"], 1)]
    return "\n\n".join(parts) + "\n\nWrite the RULE line and the python transform function."


def build_prompts(n_long_tokens: int) -> tuple[list, list]:
    tasks = json.load(open(CFG["tasks_path"]))
    keys = sorted(tasks)
    short = [[{"role": "system", "content": SYSTEM}, {"role": "user", "content": task_prompt(tasks[k])}] for k in keys[:64]]
    # long context: a task plus earlier "notes" (other tasks' grids) up to ~n_long_tokens (est. 1.9 chars/token)
    longp = []
    for j, k in enumerate(keys[:32]):
        notes, i = [], j + 1
        while sum(len(x) for x in notes) / 1.9 < n_long_tokens:
            notes.append(task_prompt(tasks[keys[i % len(keys)]]))
            i += 1
        longp.append([{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": "Reference notes:\n" + "\n\n".join(notes) + "\n\nPuzzle:\n" + task_prompt(tasks[k])}])
    return short, longp


# ============================================================================ server control
def server_env(vpy: str, pythonpath: str | None = None, gpus: str = "0,1,2,3", extra: dict | None = None) -> dict:
    home = f"/tmp/benchhome_{gpus.replace(',', '')}"
    os.makedirs(home + "/tmp", exist_ok=True)
    keep_ld = [p for p in os.environ.get("LD_LIBRARY_PATH", "").split(":") if p and "python3.11" not in p and "dist-packages" not in p]
    env = {"PATH": f"{os.path.dirname(vpy)}:/usr/local/cuda/bin:/usr/bin:/bin", "HOME": home,
           "LD_LIBRARY_PATH": ":".join(keep_ld + ["/usr/local/nvidia/lib64", "/usr/local/nvidia/lib"]),
           "LIBRARY_PATH": "/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs",
           "CUDA_VISIBLE_DEVICES": gpus, "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
           "VLLM_USE_FLASHINFER_SAMPLER": "0", "VLLM_NO_USAGE_STATS": "1", "PYTHONNOUSERSITE": "1",
           "TMPDIR": home + "/tmp", "XDG_CACHE_HOME": home + "/.cache", "TRITON_CACHE_DIR": home + "/triton",
           "TORCHINDUCTOR_CACHE_DIR": home + "/inductor", "VLLM_CACHE_ROOT": home + "/vllm",
           "TOKENIZERS_PARALLELISM": "false", "OMP_NUM_THREADS": "8", "PYTHONUNBUFFERED": "1"}
    if pythonpath:
        env["PYTHONPATH"] = pythonpath
    if CFG.get("mock"):
        env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    env.update(extra or {})  # per-setup settings, e.g. VLLM_ATTENTION_BACKEND
    return env


def gpu_pids() -> list:
    if CFG.get("mock"):
        return []
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
                             capture_output=True, text=True, timeout=30).stdout
        return [int(x) for x in out.split() if x.strip().isdigit()]
    except Exception:
        return []


def stop(proc):
    if proc is None:
        return
    for sig in (signal.SIGTERM, getattr(signal, "SIGKILL", signal.SIGTERM)):
        try:
            os.killpg(proc.pid, sig) if os.name == "posix" else proc.kill()
        except Exception:
            pass
        try:
            proc.wait(timeout=45)
        except Exception:
            pass
    t = time.time()
    while gpu_pids() and time.time() - t < 150:
        if time.time() - t > 40:
            for pid in gpu_pids():
                try:
                    os.kill(pid, getattr(signal, "SIGKILL", signal.SIGTERM))
                except Exception:
                    pass
        time.sleep(3)


def get(url: str, timeout: float = 5.0):
    return urllib.request.urlopen(url, timeout=timeout).read()


def chat(port: int, messages, max_tokens: int, timeout: float, thinking=True, ignore_eos=True) -> dict:
    body = {"model": "qwen", "messages": messages, "max_tokens": max_tokens, "temperature": 1.0 if thinking else 0,
            "top_p": 0.95, "top_k": 20, "ignore_eos": ignore_eos,
            "chat_template_kwargs": {"enable_thinking": thinking, **({"reasoning_effort": "xhigh"} if thinking else {})}}
    req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/chat/completions", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    t = time.time()
    r = json.loads(urllib.request.urlopen(req, timeout=timeout).read())
    u = r.get("usage") or {}
    msg = (r.get("choices") or [{}])[0].get("message") or {}
    return {"completion": int(u.get("completion_tokens") or 0), "prompt": int(u.get("prompt_tokens") or 0),
            "s": time.time() - t, "text": (msg.get("content") or "") + (msg.get("reasoning_content") or msg.get("reasoning") or "")}


def start(setup: dict, vpy: str, port: int, logp: str, pythonpath: str | None = None, gpus: str = "0,1,2,3",
          wait: bool = True):
    try:  # a server that survived an earlier stop would answer the readiness check and be measured instead
        get(f"http://127.0.0.1:{port}/v1/models", timeout=2)
        return None, f"port {port} already serves an API (stale server) - not starting"
    except Exception:
        pass
    cmd = [x.replace("{PY}", vpy).replace("{MODEL}", setup["model_dir"]).replace("{PORT}", str(port)) for x in setup["cmd"]]
    log(f"[{setup['name']}] start on GPUs {gpus}: {' '.join(cmd)[:400]}")
    p = subprocess.Popen(cmd, env=server_env(vpy, pythonpath, gpus, setup.get("env")), stdout=open(logp, "w"),
                         stderr=subprocess.STDOUT, start_new_session=True)
    return (p, wait_ready(p, port)) if wait else (p, None)


def wait_ready(p, port: int) -> str | None:
    t = time.time()
    while time.time() - t < CFG.get("start_timeout_s", 1200):
        if p.poll() is not None:
            return f"exited rc={p.poll()} after {time.time() - t:.0f}s"
        try:
            get(f"http://127.0.0.1:{port}/v1/models")
            return None
        except Exception:
            time.sleep(5)
    return f"not ready after {time.time() - t:.0f}s"


# ============================================================================ server-log metrics
NUM = r"([\d.]+)"


def log_metrics(txt: str) -> dict:
    """Generation throughput / running requests / KV usage / speculative acceptance from vLLM or SGLang logs."""
    gen, run, kv, acc, alen, dec = [], [], [], [], [], []
    for line in txt.splitlines():
        m = re.search(r"Avg generation throughput: " + NUM + r" tokens/s, Running: (\d+) reqs", line)  # vLLM
        if m:
            gen.append(float(m.group(1)))
            run.append(int(m.group(2)))
            pt = re.search(r"Avg prompt throughput: " + NUM, line)
            if pt and float(pt.group(1)) == 0.0:  # a pure-decode interval (no prefill in it)
                dec.append(float(m.group(1)))
            k = re.search(r"KV cache usage: " + NUM + "%", line)
            if k:
                kv.append(float(k.group(1)))
            continue
        m = re.search(r"gen throughput \(token/s\): " + NUM, line)  # SGLang decode batch line
        if m:
            gen.append(float(m.group(1)))
            r = re.search(r"#running-req: (\d+)", line)
            if r:
                run.append(int(r.group(1)))
            k = re.search(r"token usage: " + NUM, line)
            if k:
                kv.append(100 * float(k.group(1)))
            a = re.search(r"accept len: " + NUM, line)
            if a:
                alen.append(float(a.group(1)))
            continue
        m = re.search(r"Avg Draft acceptance rate: " + NUM + "%", line)  # vLLM spec decode
        if m:
            acc.append(float(m.group(1)))
        m = re.search(r"Mean acceptance length: " + NUM, line)
        if m:
            alen.append(float(m.group(1)))
    return {"gen": gen, "run": run, "kv": kv, "acc": acc, "alen": alen, "dec": dec}


def steady(vals: list, frac: float = 0.25) -> float | None:
    """Median of the samples after dropping the first `frac` (prefill ramp) and zero samples."""
    v = [x for x in vals[int(len(vals) * frac):] if x > 0]
    return round(statistics.median(v), 1) if v else None


def wave(ports: list, prompts: list, n: int, max_tokens: int, logps: list, label: str) -> dict:
    """n concurrent fixed-length generations spread round-robin over the setup's servers. Throughput per server
    comes from its own log; 'decode' = median of pure-decode intervals (no prefill), summed over servers."""
    offs = [os.path.getsize(lp) for lp in logps]
    t = time.time()
    timeout = CFG.get("wave_timeout_s", 900)

    def one(i):
        try:
            return chat(ports[i % len(ports)], prompts[i % len(prompts)], max_tokens, timeout)
        except Exception as e:
            return {"error": repr(e)[:200]}
    with ThreadPoolExecutor(n) as ex:
        res = list(ex.map(one, range(n)))
    wall = time.time() - t
    mets = []
    for lp, off in zip(logps, offs):
        with open(lp, "rb") as f:
            f.seek(off)
            mets.append(log_metrics(f.read().decode(errors="ignore")))
    ok = [r for r in res if "completion" in r]
    toks = sum(r["completion"] for r in ok)

    def total(key, frac=0.25):
        vals = [steady(m[key], frac) for m in mets]
        return round(sum(v for v in vals if v), 1) if any(vals) else None
    out = {"label": label, "n": n, "max_tokens": max_tokens, "ok": len(ok), "errors": [r["error"] for r in res if "error" in r][:3],
           "tokens": toks, "wall_s": round(wall, 1), "client_tok_s": round(toks / wall, 1) if wall else 0,
           "prompt_tokens_mean": round(statistics.mean([r["prompt"] for r in ok]), 0) if ok else None,
           "server_gen_tok_s": total("gen"), "decode_tok_s": total("dec", 0.0),
           "server_gen_peak": round(sum(max(m["gen"]) for m in mets if m["gen"]), 1) if any(m["gen"] for m in mets) else None,
           "running_median": total("run"), "kv_max_pct": max((max(m["kv"]) for m in mets if m["kv"]), default=None),
           "spec_accept_pct": steady([x for m in mets for x in m["acc"]], 0.0),
           "spec_accept_len": steady([x for m in mets for x in m["alen"]], 0.0)}
    log(f"  wave {label}: n={n} x {max_tokens} tok -> decode {out['decode_tok_s']} tok/s, server {out['server_gen_tok_s']} "
        f"(peak {out['server_gen_peak']}), client {out['client_tok_s']} tok/s, wall {out['wall_s']}s, ok {len(ok)}/{n}, "
        f"prompt {out['prompt_tokens_mean']}, kv max {out['kv_max_pct']}%, accept {out['spec_accept_pct']}% / len {out['spec_accept_len']}")
    return out


def run_setup(setup: dict, vpy: str, prompts_short: list, prompts_long: list, idx: int = 0, pythonpath: str | None = None) -> dict:
    """Start the setup's server(s) - one on all 4 GPUs, or several 'parts' (e.g. 2 x TP2 on GPUs 0,1 and 2,3) - try
    alternative commands if a start fails, sanity-check every server, then run the waves over all of them."""
    parts = setup.get("parts") or [{"gpus": "0,1,2,3"}]
    base_port = setup.get("port") or 18100 + 10 * idx  # own ports per setup
    ports = [base_port + j for j in range(len(parts))]
    logps = [os.path.join(CFG["work_dir"], f"server_{setup['name']}" + (f"_{j}" if len(parts) > 1 else "") + ".log")
             for j in range(len(parts))]
    r = {"name": setup["name"], "stack": setup["stack"], "model": os.path.basename(setup["model_dir"].rstrip("/")), "ok": False,
         "parts": len(parts)}
    t = time.time()
    procs = []
    try:
        cmds = [setup["cmd"]] + list(setup.get("alts", []))
        err, tried = None, []
        while cmds:
            cmd = cmds.pop(0)
            procs, err = [], None
            for part, port, lp in zip(parts, ports, logps):  # start all parts in parallel
                p, e = start({**setup, "cmd": cmd}, vpy, port, lp, pythonpath, part["gpus"], wait=False)
                procs.append(p)
                err = err or e
            if not err:
                for p, port in zip(procs, ports):
                    err = err or wait_ready(p, port)
            tried.append({"cmd": " ".join(cmd)[-300:], "error": err})
            if not err:
                r["cmd_used"] = " ".join(cmd)[-400:]
                break
            tail = "\n".join(open(lp, errors="ignore").read()[-3000:] for lp in logps if os.path.exists(lp))
            r["log_tail"] = tail[-5000:]
            for p in procs:
                stop(p)
            procs = []
            bad = re.search(r"unrecognized arguments: (\S+)", tail)
            if bad and bad.group(1) in cmd:  # retry once without a flag this version does not know
                j = cmd.index(bad.group(1))
                drop = 2 if j + 1 < len(cmd) and not cmd[j + 1].startswith("--") else 1
                cmds.insert(0, cmd[:j] + cmd[j + drop:])
                log(f"[{setup['name']}] retry without {bad.group(1)}")
            elif cmds:
                log(f"[{setup['name']}] start failed ({err}); trying the next alternative")
        r["tried"] = tried
        r["start_s"] = round(time.time() - t)
        if err:
            r["error"] = err
            log(f"[{setup['name']}] FAILED to start: {err}")
            return r
        for port in ports:
            s = chat(port, [{"role": "user", "content": "What is 17 + 25? Reply with only the number."}], 24, 300,
                     thinking=False, ignore_eos=False)
            r["sanity"] = s["text"].strip()[:60]
            if "42" not in s["text"]:
                r["error"] = f"sanity answer {r['sanity']!r} on port {port}"
                log(f"[{setup['name']}] FAILED sanity: {r['sanity']!r}")
                return r
        txt = "\n".join(open(lp, errors="ignore").read() for lp in logps)
        mx = re.search(r"Maximum concurrency for ([\d,]+) tokens per request: ([\d.]+)x", txt)
        kvl = re.search(r"GPU KV cache size: ([\d,]+) tokens", txt) or re.search(r"max_total_num_tokens=(\d+)", txt)
        r["kv_line"] = kvl.group(0)[:120] if kvl else None
        r["concurrency_line"] = mx.group(0)[:120] if mx else None
        log(f"[{setup['name']}] READY in {r['start_s']}s ({len(parts)} server(s)); {r['kv_line']}; {r['concurrency_line']}")
        r["waves"] = []
        for w in setup.get("waves", CFG["waves"]):
            prompts = prompts_long if w.get("long") else prompts_short
            r["waves"].append(wave(ports, prompts, w["n"], w["max_tokens"], logps, w["label"]))
            save()
        r["ok"] = any((w.get("server_gen_tok_s") or w.get("client_tok_s") or 0) > 0 for w in r["waves"])
    except Exception as e:
        r["error"] = repr(e)[:400]
        log(f"[{setup['name']}] ERROR {e!r}")
    finally:
        r["total_s"] = round(time.time() - t)
        for p in procs:
            stop(p)
    return r


# ============================================================================ main
def main():
    CFG.update(json.load(open(sys.argv[1])))
    deadline = T0 + CFG["budget_s"]
    py = setup_python()
    short, longp = build_prompts(CFG.get("long_tokens", 16000))
    stacks = {s["name"]: s for s in CFG["stacks"]}
    installs: dict = {}

    def do_install(name):
        installs[name] = install(py, stacks[name])
        RES["installs"][name] = {k: v for k, v in installs[name].items() if k != "pip_tail" or not installs[name]["ok"]}
        log(f"install {name}: {'OK' if installs[name]['ok'] else 'FAILED'} in {installs[name].get('s')}s "
            f"{installs[name].get('import', installs[name].get('error', ''))[:160]}")
        save()

    # weights into the page cache while the first stack installs; later stacks install in the background
    models = sorted({s["model_dir"] for s in CFG["setups"]})
    if not CFG.get("mock"):
        threading.Thread(target=warm, args=(models,), daemon=True).start()
    order = []
    for s in CFG["setups"]:
        if s["stack"] not in order:
            order.append(s["stack"])
    do_install(order[0])

    def installer(names):  # later stacks, one at a time, while the GPUs benchmark the earlier ones
        for name in names:
            t = time.time()
            while (not CFG.get("mock") and shutil.disk_usage("/tmp").free / 2 ** 30 < stacks[name].get("need_gb", 14)
                   and time.time() - t < 1800):
                time.sleep(15)
            do_install(name)
    threading.Thread(target=installer, args=(order[1:],), daemon=True).start()
    last_of = {s["stack"]: i for i, s in enumerate(CFG["setups"])}
    for i, setup in enumerate(CFG["setups"]):
        name = setup["stack"]
        left = deadline - time.time()
        if left < CFG.get("min_setup_s", 900):
            log(f"[{setup['name']}] skipped: {left / 60:.0f} min left")
            RES["configs"].append({"name": setup["name"], "ok": False, "error": "skipped (time budget)"})
        else:
            t = time.time()
            while name not in installs and time.time() - t < max(1, left - CFG.get("min_setup_s", 900)):
                time.sleep(5)
            st = installs.get(name)
            if not st or not st.get("ok"):
                RES["configs"].append({"name": setup["name"], "ok": False, "error": f"stack {name} not installed"})
                log(f"[{setup['name']}] skipped: stack {name} not installed")
            else:
                RES["configs"].append(run_setup(setup, st["python"], short, longp, i, st.get("pythonpath")))
        save()
        if last_of[name] == i and not CFG.get("mock") and any(n not in installs for n in order):
            shutil.rmtree(f"/tmp/venv_{name}", ignore_errors=True)  # free disk for the stacks still installing
            log(f"removed venv {name} (free {shutil.disk_usage('/tmp').free / 2 ** 30:.0f} GB)")
    # summary table
    print("\n" + "=" * 118 + "\n SPEED TEST  (server-side steady generation tok/s; client = tokens / wall incl. prefill)\n" + "=" * 118)
    best = None
    for c in RES["configs"]:
        if not c.get("waves"):
            print(f" {c['name']:<34} FAIL  {str(c.get('error'))[:70]}")
            continue
        cells = "  ".join(f"{w['label']}: dec {w.get('decode_tok_s')} / srv {w['server_gen_tok_s']} / cli {w['client_tok_s']} "
                          f"(kv {w['kv_max_pct']}%, acc {w['spec_accept_pct']})" for w in c["waves"])
        print(f" {c['name']:<34} start {c.get('start_s')}s  {cells}")
        # ranking metric: pure-decode speed in the long-context wave (what xhigh answers live in), else any wave
        longw = [w for w in c["waves"] if w["label"].startswith("long")]
        w0 = longw[0] if longw else c["waves"][0]
        v = w0.get("decode_tok_s") or w0.get("server_gen_tok_s") or w0.get("client_tok_s") or 0
        if best is None or v > best[1]:
            best = (c["name"], v)
    RES["best"] = best
    RES["gate_500"] = bool(best and best[1] >= 500)
    print(f"\n BEST (long-context decode): {best}  |  >= 500 tok/s gate: {'PASS' if RES['gate_500'] else 'FAIL'}  |  "
          f"total {(time.time() - T0) / 60:.1f} min")
    save()


if __name__ == "__main__":
    main()


In [ ]:
import glob, json, os, subprocess, sys, time
T0 = time.time()
def model_dir(tag):
    for c in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        if tag in c.lower() and "Qwen3_5" in open(c).read() and glob.glob(os.path.join(os.path.dirname(c), "*.safetensors")):
            return os.path.dirname(c)
    return None
dirs = {"FP8": model_dir("fp8-repacked"), "W4": model_dir("gptq-w4a16")}
print("models:", dirs)
setups = []
for s in json.loads('[{"name": "V1 w4 mtp1 16seq (baseline)", "stack": "vllm030", "model": "W4", "cmd": ["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 1}"]}, {"name": "V2 w4 mtp3 no-fp8kv", "stack": "vllm030", "model": "W4", "cmd": ["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 3}"], "alts": [["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 2}"]]}, {"name": "V3 w4 mtp1 batch8192", "stack": "vllm030", "model": "W4", "cmd": ["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 1}", "--max-num-batched-tokens", "8192"]}, {"name": "V4 w4 mtp1 fp8kv triton", "stack": "vllm030", "model": "W4", "env": {"VLLM_ATTENTION_BACKEND": "TRITON_ATTN"}, "cmd": ["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 1}", "--kv-cache-dtype", "fp8", "--attention-backend", "TRITON_ATTN"], "alts": [["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "4", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "16", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 1}", "--kv-cache-dtype", "fp8"]]}, {"name": "V5 w4 mtp1 2xTP2", "stack": "vllm030", "model": "W4", "parts": [{"gpus": "0,1"}, {"gpus": "2,3"}], "cmd": ["{PY}", "-m", "vllm.entrypoints.openai.api_server", "--model", "{MODEL}", "--served-model-name", "qwen", "--port", "{PORT}", "--host", "127.0.0.1", "--tensor-parallel-size", "2", "--max-model-len", "65536", "--gpu-memory-utilization", "0.9", "--max-num-seqs", "8", "--no-enable-prefix-caching", "--reasoning-parser", "qwen3", "--dtype", "bfloat16", "--language-model-only", "--speculative-config", "{\\"method\\": \\"mtp\\", \\"num_speculative_tokens\\": 1}"]}]'):
    d = dirs.get(s["model"])
    if d:
        setups.append({**s, "model_dir": d})
    else:
        print("no model for", s["name"])
cfg = {"work_dir": "/kaggle/working", "budget_s": 95 * 60, "min_setup_s": 15 * 60, "start_timeout_s": 1200,
       "wave_timeout_s": 900, "long_tokens": 13000,
       "tasks_path": "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json",
       "stacks": json.loads('[{"name": "vllm030", "wheel": "/kaggle/input/**/vllm-0.30.0*.whl", "pkgs": ["vllm==0.30.0"], "need_gb": 14, "import_check": "import vllm, torch; print(\'vllm\', vllm.__version__, \'torch\', torch.__version__)"}]'), "setups": setups, "waves": json.loads('[{"label": "short16", "n": 16, "max_tokens": 1536}, {"label": "long16", "n": 16, "max_tokens": 1536, "long": true}]')}
json.dump(cfg, open("/kaggle/working/bench_cfg.json", "w"), indent=1)
try:
    p = subprocess.run([sys.executable, "bench.py", "/kaggle/working/bench_cfg.json"], timeout=110 * 60)
    print("bench rc", p.returncode, f"| {(time.time() - T0) / 60:.1f} min")
except Exception as e:  # partial results are already in /kaggle/working/bench_results.json
    print("bench stopped:", repr(e))
    try:
        subprocess.run(["pkill", "-9", "-f", "/tmp/venv_"], capture_output=True, timeout=30)
    except Exception:
        pass
